## 0. Kernel setup (run in a terminal, not in this notebook)

Before launching this notebook, create and select a conda environment kernel (`2ndWorkshop`).

### Purdue Gilbreth cluster

```bash
module load conda
conda-env-mod create -n ENV_NAME_HERE -j
module use $HOME/privatemodules
module load conda-env/ENV_NAME_HERE-py3.10.11
```

Replace `ENV_NAME_HERE` with your environment name (`2ndWorkshop`), then select the matching kernel in Jupyter before running the cells below.

Check the env and kernel were created:
```bash
conda env list
jupyter kernelspec list
```

### Local machine (conda)

```bash
conda create -n 2ndWorkshop python=3.10 pip -y
conda activate 2ndWorkshop
pip install ipykernel
python -m ipykernel install --user --name 2ndWorkshop --display-name "Python (2ndWorkshop)"
```

Select the `Python (2ndWorkshop)` kernel, run the install cell below once, then restart the kernel.

**Note:** This notebook (Part 1) only needs local Hugging Face models — no API keys or
external servers required. **Part 2** (`Workshop2_Part2_Agent.ipynb`) is where you'll
need an LLM backend that supports tool calling (Ollama, OpenAI, or Purdue GenAI).


# Workshop 2, Part 1: Retrieval-Augmented Generation (RAG)

This is **Part 1 of 2** for Workshop 2. Build a RAG pipeline over a small Purdue course
knowledge base: embed documents, index them with FAISS, retrieve the most relevant
passages for a query, and generate a grounded answer with a local LLM.

**What you'll do:**
1. Load and inspect the course knowledge base
2. Embed documents and build a FAISS index
3. Retrieve the top-k relevant passages for a query
4. Build a chat-templated RAG prompt (same trick as Workshop 1)
5. Load a small local instruct model and answer real questions with cited sources

**Part 2** (`Workshop2_Part2_Agent.ipynb`) reuses this same retrieval approach, but wraps
it in tools an autonomous LangGraph agent can call — the Boilermaker TA.


## 0. Install dependencies

Run once, then restart the kernel.

In [3]:
! pip install torch transformers accelerate sentence-transformers faiss-cpu


  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.14.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
  Using cached http

---
# Part A: Retrieval-Augmented Generation (RAG)

RAG = embed documents → store in a vector index → at query time, retrieve the most relevant chunks → pass them as context to the LLM.

## A1. Imports and data loading

In [4]:
import os
import json
from pathlib import Path
from typing import List

import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "boilermaker_ta_data"

print("Data directory:", DATA_DIR)
print("Files:", list(DATA_DIR.iterdir()) if DATA_DIR.exists() else "NOT FOUND")

/Users/elhambarezi/miniconda3/envs/2ndWorkshop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directory: /Users/elhambarezi/Desktop/next workshop/2ndWorkshop/boilermaker_ta_data
Files: [PosixPath('/Users/elhambarezi/Desktop/next workshop/2ndWorkshop/boilermaker_ta_data/purdue_calendar.json'), PosixPath('/Users/elhambarezi/Desktop/next workshop/2ndWorkshop/boilermaker_ta_data/knowledge_base.json')]


## A2. Load the knowledge base corpus

In [5]:
def _load_corpus():
    path = DATA_DIR / "knowledge_base.json"
    if not path.exists():
        # Fallback so the notebook runs even without the data file
        return [
            {
                "title": "Fallback: RAG Intro",
                "text": "This is a fallback document. Add boilermaker_ta_data/knowledge_base.json for workshop data.",
            }
        ]
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


CORPUS = _load_corpus()
print(f"Loaded {len(CORPUS)} documents")
print("First doc:", CORPUS[0]["title"])

Loaded 3 documents
First doc: CS 182 Syllabus


## A3. Build the FAISS retriever

Steps:
1. **Embed** each document with `all-MiniLM-L6-v2` (384-dim vectors)  
2. **L2-normalize** vectors so inner product = cosine similarity  
3. **Index** in a FAISS `IndexFlatIP` (brute-force inner product search)  

In production, save/load the index from disk to avoid re-embedding on every startup.

**Note on chunking:** our course documents are short (a paragraph each), so we embed
them whole. Real corpora (PDFs, wikis) are usually split into ~200-500 token chunks
first, so retrieval returns a focused excerpt instead of an entire long document.


In [6]:
def build_retriever(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    embedder  = SentenceTransformer(model_name)
    texts     = [doc["text"] for doc in CORPUS]
    metadatas = [{"title": doc.get("title") or f"doc{i}"} for i, doc in enumerate(CORPUS)]

    embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)

    faiss.normalize_L2(embeddings)          # normalize so dot product = cosine sim
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return {"embedder": embedder, "index": index, "texts": texts, "metadatas": metadatas}


retriever = build_retriever()
print(f"FAISS index has {retriever['index'].ntotal} vectors of dim {retriever['index'].d}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.36it/s]

FAISS index has 3 vectors of dim 384


## A4. Retrieve documents for a query

In [7]:
def retrieve_docs(retriever, query: str, k: int = 3):
    q_emb = retriever["embedder"].encode(query, convert_to_numpy=True)
    if q_emb.ndim == 1:
        q_emb = q_emb.reshape(1, -1)
    faiss.normalize_L2(q_emb)
    scores, ids = retriever["index"].search(q_emb, min(k, len(retriever["texts"])))
    return [
        {
            "text":     retriever["texts"][idx],
            "metadata": retriever["metadatas"][idx],
            "score":    float(scores[0][i]),
        }
        for i, idx in enumerate(ids[0])
    ]


# Test retrieval
query = "When are the office hours for this course?"
docs  = retrieve_docs(retriever, query, k=3)
for d in docs:
    print(f"[{d['score']:.3f}] {d['metadata']['title']}")
    print("  ", d["text"][:120], "...\n")

[0.439] Course Resources
   The course uses GitHub Classroom, Piazza for questions, and a shared lecture notes repository. Instructors recommend sta ...

[0.428] Exam Policies
   Purdue exam weeks are in mid-October and mid-November. Makeup exams require prior approval. Students are encouraged to s ...

[0.397] CS 182 Syllabus
   CS 182 is a Purdue undergraduate course focused on software development. Assignments are due on Wednesdays at 11:59 PM.  ...



## A5. Build a RAG prompt and generate an answer

The retrieved context is inserted into the prompt so the LLM can ground its answer in
retrieved facts instead of guessing.

Just like Workshop 1's `build_messages` + `apply_chat_template`, we render a
`system` / `user` message pair through the model's native chat template — instruct
models answer far more reliably this way than from a raw concatenated string.

The preview below can't use the real chat template yet because the tokenizer is loaded
in **A6**, one section from now — it prints the plain-text fallback. **A7** calls
`make_prompt` again with the loaded tokenizer, so the actual queries do get the
chat-templated version.


In [8]:
def make_prompt(query: str, context: str, tokenizer=None) -> str:
    # Same trick as Workshop 1: instruct models follow instructions far more reliably
    # when the prompt is rendered through their native chat template instead of a
    # raw string glued together with "Context:" / "Question:" labels.
    messages = [
        {
            "role": "system",
            "content": "You are a course TA. Answer using only the provided context. "
            "If the answer isn't in the context, say you don't know.",
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    if tokenizer is not None:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    # Fallback for base (non-chat) models, or when no tokenizer is available yet
    return f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer in a concise, factual way."


# Preview without a tokenizer (fallback format) — A7 renders the real chat-templated version
context = "\n\n".join([d["text"] for d in docs])
prompt  = make_prompt(query, context)
print(prompt)


Context:
The course uses GitHub Classroom, Piazza for questions, and a shared lecture notes repository. Instructors recommend starting project planning at least three weeks before each milestone.

Purdue exam weeks are in mid-October and mid-November. Makeup exams require prior approval. Students are encouraged to schedule study groups at least one week before the exam.

CS 182 is a Purdue undergraduate course focused on software development. Assignments are due on Wednesdays at 11:59 PM. Office hours are Tuesdays 3-5 PM and Thursdays 2-4 PM. The final project presentation is scheduled in the last week of classes.

Question: When are the office hours for this course?

Answer in a concise, factual way.


## A6. Load the local LLM

Same default as Workshop 1 — **`Qwen/Qwen2.5-0.5B-Instruct`** — small enough to run on a
laptop CPU in a few seconds, so the RAG demo is reliable for everyone in the room.

`LOCAL_MODEL` env var lets you swap models without changing code.
`device_map="auto"` places layers on GPU if available, CPU otherwise.
`bfloat16` is only requested when CUDA is available — avoids issues on CPU-only machines.


In [9]:
def get_llm():
    # Swap example: LOCAL_MODEL="mistralai/Mistral-7B-Instruct-v0.3" (needs a GPU)
    local_model_name = os.environ.get("LOCAL_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
    tokenizer    = AutoTokenizer.from_pretrained(local_model_name)
    model_kwargs = {"device_map": "auto"}
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.bfloat16   # bfloat16 only safe with CUDA

    local_model = AutoModelForCausalLM.from_pretrained(local_model_name, **model_kwargs)
    return pipeline("text-generation", model=local_model, tokenizer=tokenizer, max_new_tokens=256)


print("Loading LLM... (a few seconds on CPU with the default model)")
llm = get_llm()
print("LLM ready.")


Loading LLM... (a few seconds on CPU with the default model)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 52156.78it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready.


## A7. Run RAG queries end-to-end

In [10]:
def answer_queries(queries: List[str], llm, retriever):
    for query in queries:
        print(f"\n=== QUERY: {query}\n")
        docs    = retrieve_docs(retriever, query, k=3)
        context = "\n\n".join([d["text"] for d in docs])
        # llm.tokenizer: transformers pipelines expose the tokenizer they were built
        # with, so we can render the same chat template used during A6's load.
        prompt  = make_prompt(query, context, tokenizer=llm.tokenizer)
        output  = llm(prompt, return_full_text=False)[0]["generated_text"]
        print("Answer:\n", output)
        print("Sources:\n", [d["metadata"] for d in docs])


queries = [
    "When are the office hours for this course?",
    "When are assignments due and what is the usual deadline?",
    "When are the exam weeks scheduled?",
    "What tools and resources does the course recommend for projects?",
    "When is the final project presentation scheduled?",
]

answer_queries(queries, llm, retriever)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== QUERY: When are the office hours for this course?



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Office hours are Tuesdays 3-5 PM and Thursdays 2-4 PM.
Sources:
 [{'title': 'Course Resources'}, {'title': 'Exam Policies'}, {'title': 'CS 182 Syllabus'}]

=== QUERY: When are assignments due and what is the usual deadline?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Assignments are due on Wednesdays at 11:59 PM.
Sources:
 [{'title': 'Course Resources'}, {'title': 'CS 182 Syllabus'}, {'title': 'Exam Policies'}]

=== QUERY: When are the exam weeks scheduled?

Answer:
 Mid-October and Mid-November
Sources:
 [{'title': 'Exam Policies'}, {'title': 'Course Resources'}, {'title': 'CS 182 Syllabus'}]

=== QUERY: What tools and resources does the course recommend for projects?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 The course recommends using GitHub Classroom for project planning and Piazza for questions.
Sources:
 [{'title': 'Course Resources'}, {'title': 'CS 182 Syllabus'}, {'title': 'Exam Policies'}]

=== QUERY: When is the final project presentation scheduled?

Answer:
 The final project presentation is scheduled in the last week of classes.
Sources:
 [{'title': 'Course Resources'}, {'title': 'CS 182 Syllabus'}, {'title': 'Exam Policies'}]


---
## Next: Part 2 — Agentic AI

Open **`Workshop2_Part2_Agent.ipynb`** to continue. It reuses the same embeddings + FAISS
approach from this notebook, but wraps it in tools that an autonomous LangGraph agent
decides when to call — plus a calendar tool and a notification-writing tool. That's the
Boilermaker TA.
